# depthfly — depth 카메라(Gemini 2L) 하나 + 초파리 커넥톰만

RGB·LiDAR·CNN·MLP 없음. **depth 가까움 영상 3프레임(30 Hz)** → 합성 커넥톰 시각엽(T4/T5 → HS/VS·LPLC2·LC·L3 → 하행 뉴런 48) → 선형 읽기 → 조향·속도.
차량·보상·레이싱라인(critic 전용)은 mapless40 과 같다.

**순서:** ① → ② → ③ → ④ (9/9 통과 확인) → ⑤ → ⑥.
**세션이 끊기면:** ① ~ ③ 과 ⑥ 학습 셀만 다시 실행 (`--resume auto` 가 드라이브의 `last_model.zip` 에서 이어 감).

관측 정의 v2 (행을 지평선 ±8° 에 집중, 거리 비례 높이 판정, lamina 상대 변화). **v2 이전 모델과 호환 안 됨** → 저장 폴더 `depthfly_v2`.


## ① GPU 확인

In [ ]:
!nvidia-smi -L || echo "GPU 없음 → 런타임 유형을 T4 GPU 로 바꾸세요"
!python --version

## ② 구글 드라이브 연결

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
RUN_DIR = '/content/drive/MyDrive/mapless40_runs/depthfly_v2'
import os; os.makedirs(RUN_DIR, exist_ok=True); print(RUN_DIR)


## ③ 코드 받기 + 패키지 설치

In [ ]:
%cd /content
!test -d 2027_F1tenth_test || git clone -q --depth 1 -b feat/mapless40-asym-sac https://github.com/tkddn647-ship-it/2027_F1tenth_test
%cd /content/2027_F1tenth_test
!git pull -q origin feat/mapless40-asym-sac
!git log --oneline -1
!pip -q install "stable-baselines3>=2.3" "gymnasium>=0.29" scipy pyyaml pillow tensorboard

## ④ 테스트 (numpy 7개 + torch 2개 → `9/9 passed` 가 나와야 함)

In [ ]:
!python -m camera.depthfly.tests


## ⑤ depth 파리 눈이 보는 것 (학습 전, pure pursuit 로 주행)

In [ ]:
!python -m camera.depthfly.viz --map ifac --seconds 6 --obstacles 1 --out /content/depthfly_eye.gif
from IPython.display import Image, display
display(Image('/content/depthfly_eye.gif'))


## ⑥ 학습 (30 Hz, 과거 3프레임, 속도 2~8 m/s, actor = 커넥톰 + 선형 읽기)

150분 뒤 스스로 저장하고 멈춘다. 평가는 5만 스텝마다 (`best_model.zip`, `best.json` 갱신). 다시 실행하면 이어서 학습.

In [ ]:
MAPS = 'ifac:3,roboracer_0817:3,Spielberg:1,Silverstone:1,Monza:1'
EVAL_MAPS = 'ifac,roboracer_0817'
STEPS_PER_SESSION = 1_000_000
TIME_LIMIT_MIN = 150
N_ENVS = 4
V_MAX = 8.0        # 최고 속도 [m/s] (최저 2). 평가·GIF 셀도 같은 값을 쓴다
BRAKE = ''         # 능동 브레이크로 학습하려면 '--brake' (실차는 지금 타력 감속만)

!python -m camera.depthfly.train --maps {MAPS} --eval-maps {EVAL_MAPS} \
   --timesteps {STEPS_PER_SESSION} --n-envs {N_ENVS} --subproc --device cuda \
   --time-limit-min {TIME_LIMIT_MIN} --save-dir "{RUN_DIR}" --resume auto \
   --v-max {V_MAX} {BRAKE}


## ⑥-2 진행 확인 (학습 셀이 끝난 뒤, 또는 다른 세션에서)

In [ ]:
import json, os
for f in ('best.json',):
    q = os.path.join(RUN_DIR, f)
    print(f, '→', json.load(open(q)) if os.path.exists(q) else '아직 없음')
print(sorted(os.listdir(RUN_DIR)))


## ⑦ 평가 (장애물 없이 / 2개)

In [ ]:
!python -m camera.depthfly.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,roboracer_0817 --spawns 4 --laps 3 --obstacles 0 --v-max {V_MAX} {BRAKE}
!python -m camera.depthfly.evaluate --model "{RUN_DIR}/best_model.zip" --maps ifac,roboracer_0817 --spawns 4 --laps 3 --obstacles 2 --v-max {V_MAX} {BRAKE}


## ⑧ 학습된 정책 주행 + 회로 활동 GIF

In [ ]:
!python -m camera.depthfly.viz --map ifac --model "{RUN_DIR}/best_model.zip" --seconds 12 --obstacles 1 --out /content/depthfly_trained.gif --v-max {V_MAX} {BRAKE}
from IPython.display import Image, display
display(Image('/content/depthfly_trained.gif'))
